# PINN Project 3: 2D Poisson Equation
## Solving PDEs on a 2D Domain

### Problem Description
The **2D Poisson equation** describes electrostatics, heat conduction, and gravity:

$$\frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} = f(x, y)$$

We solve on a square domain $[0, 1] \times [0, 1]$ with:
- Source term: $f(x, y) = -2\pi^2 \sin(\pi x) \sin(\pi y)$
- Boundary conditions: $u = 0$ on all four edges

**Exact solution**: $u(x, y) = \sin(\pi x) \sin(\pi y)$

### What's new compared to Projects 1-2?
- **2D spatial domain** instead of 1D
- **Two spatial derivatives**: $\partial^2 u/\partial x^2$ and $\partial^2 u/\partial y^2$
- **Four boundaries** instead of two
- More complex visualization (contour plots, 3D surfaces)


In [ ]:
# ============================================================
# STEP 1: Import Libraries
# ============================================================
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


In [ ]:
# ============================================================
# STEP 2: Define the Neural Network
# ============================================================

class PINN_Poisson2D(nn.Module):
    """
    PINN for the 2D Poisson equation.
    
    Input:  (x, y) — 2 spatial coordinates
    Output: u(x, y) — scalar field (e.g., temperature, potential)
    
    New feature: Residual connections (skip connections)
    - Help gradients flow through deep networks
    - Improve convergence for 2D problems
    """
    
    def __init__(self, n_hidden=5, n_neurons=64):
        super().__init__()
        
        # Input layer
        self.input_layer = nn.Linear(2, n_neurons)
        
        # Hidden layers (with residual connections)
        self.hidden_layers = nn.ModuleList([
            nn.Linear(n_neurons, n_neurons) for _ in range(n_hidden)
        ])
        
        # Output layer
        self.output_layer = nn.Linear(n_neurons, 1)
        
        # Activation function
        self.activation = nn.Tanh()
        
        # Xavier initialization
        self._init_weights()
    
    def _init_weights(self):
        """Xavier initialization for better convergence."""
        for m in [self.input_layer, self.output_layer] + list(self.hidden_layers):
            nn.init.xavier_normal_(m.weight)
            nn.init.zeros_(m.bias)
    
    def forward(self, x, y):
        """
        Forward pass with residual connections.
        
        Residual connection: output = activation(Linear(input)) + input
        This helps the network learn corrections to the identity function,
        which makes training easier for deeper networks.
        """
        # Combine inputs
        inputs = torch.cat([x, y], dim=1)
        
        # Input layer
        h = self.activation(self.input_layer(inputs))
        
        # Hidden layers with residual (skip) connections
        for layer in self.hidden_layers:
            h_new = self.activation(layer(h))
            h = h_new + h   # Residual connection: add input to output
        
        # Output layer (no activation — regression output)
        return self.output_layer(h)

model = PINN_Poisson2D(n_hidden=5, n_neurons=64).to(device)
print("Model Architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# ============================================================
# STEP 3: Physics Loss for 2D Poisson Equation
# ============================================================

def compute_pde_residual(model, x, y):
    """
    Compute the Poisson equation residual:
        d²u/dx² + d²u/dy² - f(x,y) = 0
    
    where f(x,y) = -2π² sin(πx) sin(πy)
    
    We compute:
    1. du/dx  → d²u/dx²  (second derivative w.r.t. x)
    2. du/dy  → d²u/dy²  (second derivative w.r.t. y)
    3. Laplacian = d²u/dx² + d²u/dy²
    """
    u = model(x, y)
    
    # --- Derivatives w.r.t. x ---
    du_dx = torch.autograd.grad(
        u, x, grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    
    d2u_dx2 = torch.autograd.grad(
        du_dx, x, grad_outputs=torch.ones_like(du_dx),
        create_graph=True
    )[0]
    
    # --- Derivatives w.r.t. y ---
    du_dy = torch.autograd.grad(
        u, y, grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    
    d2u_dy2 = torch.autograd.grad(
        du_dy, y, grad_outputs=torch.ones_like(du_dy),
        create_graph=True
    )[0]
    
    # Laplacian: ∇²u = d²u/dx² + d²u/dy²
    laplacian = d2u_dx2 + d2u_dy2
    
    # Source term: f(x,y) = -2π² sin(πx) sin(πy)
    f = -2 * (np.pi ** 2) * torch.sin(np.pi * x) * torch.sin(np.pi * y)
    
    # Residual: ∇²u - f = 0
    residual = laplacian - f
    
    return residual


def compute_loss(model, x_pde, y_pde, x_bc, y_bc, u_bc):
    """
    Total loss = PDE loss + Boundary loss.
    
    Args:
        x_pde, y_pde: Interior collocation points
        x_bc, y_bc: Boundary points
        u_bc: Known boundary values (all zeros for this problem)
    """
    # PDE loss at interior points
    residual = compute_pde_residual(model, x_pde, y_pde)
    loss_pde = torch.mean(residual ** 2)
    
    # Boundary condition loss
    u_pred_bc = model(x_bc, y_bc)
    loss_bc = torch.mean((u_pred_bc - u_bc) ** 2)
    
    # Weighted total loss
    total_loss = loss_pde + 10.0 * loss_bc
    
    return total_loss, loss_pde, loss_bc

print("Loss functions defined!")


In [ ]:
# ============================================================
# STEP 4: Prepare Training Points
# ============================================================

N_pde = 5000   # Interior collocation points
N_bc = 400     # Boundary points (100 per edge)

# --- Interior collocation points ---
x_pde = torch.rand(N_pde, 1, device=device, requires_grad=True)
y_pde = torch.rand(N_pde, 1, device=device, requires_grad=True)

# --- Boundary condition points ---
# We need points on all 4 edges of the unit square [0,1] x [0,1]

n_per_edge = N_bc // 4  # Points per edge

# Bottom edge: y=0, x∈[0,1]
x_b = torch.rand(n_per_edge, 1, device=device)
y_b = torch.zeros(n_per_edge, 1, device=device)

# Top edge: y=1, x∈[0,1]
x_t = torch.rand(n_per_edge, 1, device=device)
y_t = torch.ones(n_per_edge, 1, device=device)

# Left edge: x=0, y∈[0,1]
x_l = torch.zeros(n_per_edge, 1, device=device)
y_l = torch.rand(n_per_edge, 1, device=device)

# Right edge: x=1, y∈[0,1]
x_r = torch.ones(n_per_edge, 1, device=device)
y_r = torch.rand(n_per_edge, 1, device=device)

# Combine all boundary points
x_bc = torch.cat([x_b, x_t, x_l, x_r], dim=0)
y_bc = torch.cat([y_b, y_t, y_l, y_r], dim=0)
u_bc = torch.zeros(N_bc, 1, device=device)  # u=0 on all boundaries

print(f"Interior points: {N_pde}")
print(f"Boundary points: {N_bc} ({n_per_edge} per edge × 4 edges)")

# Visualize the training points
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.scatter(x_pde.detach().cpu(), y_pde.detach().cpu(), 
           s=1, alpha=0.3, c='blue', label='PDE points')
ax.scatter(x_bc.cpu(), y_bc.cpu(), 
           s=10, c='red', label='BC points')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Training Point Distribution')
ax.legend()
ax.set_aspect('equal')
plt.show()


In [ ]:
# ============================================================
# STEP 5: Training Loop
# ============================================================

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=500, factor=0.5, min_lr=1e-6
)

n_epochs = 8000
print_every = 1000

history = {'total': [], 'pde': [], 'bc': []}

print("Starting training...")
print("=" * 60)

for epoch in range(1, n_epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    # Resample interior points each epoch for better coverage
    x_pde_train = torch.rand(N_pde, 1, device=device, requires_grad=True)
    y_pde_train = torch.rand(N_pde, 1, device=device, requires_grad=True)
    
    total_loss, pde_loss, bc_loss = compute_loss(
        model, x_pde_train, y_pde_train, x_bc, y_bc, u_bc
    )
    
    total_loss.backward()
    optimizer.step()
    scheduler.step(total_loss)
    
    history['total'].append(total_loss.item())
    history['pde'].append(pde_loss.item())
    history['bc'].append(bc_loss.item())
    
    if epoch % print_every == 0 or epoch == 1:
        print(f"Epoch {epoch:5d}/{n_epochs} | "
              f"Total: {total_loss.item():.2e} | "
              f"PDE: {pde_loss.item():.2e} | "
              f"BC: {bc_loss.item():.2e}")

print("=" * 60)
print("Training complete!")


In [ ]:
# ============================================================
# STEP 6: Visualize Results
# ============================================================

# Create evaluation grid
nx, ny = 100, 100
x_grid = np.linspace(0, 1, nx)
y_grid = np.linspace(0, 1, ny)
X, Y = np.meshgrid(x_grid, y_grid)

# PINN prediction
x_flat = torch.tensor(X.flatten(), dtype=torch.float32).reshape(-1, 1).to(device)
y_flat = torch.tensor(Y.flatten(), dtype=torch.float32).reshape(-1, 1).to(device)

model.eval()
with torch.no_grad():
    u_pred = model(x_flat, y_flat).cpu().numpy().reshape(ny, nx)

# Exact solution
u_exact = np.sin(np.pi * X) * np.sin(np.pi * Y)

# Error
error = np.abs(u_pred - u_exact)

# --- Plotting ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Plot 1: PINN Prediction
c1 = axes[0, 0].contourf(X, Y, u_pred, levels=50, cmap='viridis')
plt.colorbar(c1, ax=axes[0, 0])
axes[0, 0].set_title('PINN Prediction', fontsize=14)
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[0, 0].set_aspect('equal')

# Plot 2: Exact Solution
c2 = axes[0, 1].contourf(X, Y, u_exact, levels=50, cmap='viridis')
plt.colorbar(c2, ax=axes[0, 1])
axes[0, 1].set_title('Exact Solution', fontsize=14)
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('y')
axes[0, 1].set_aspect('equal')

# Plot 3: Absolute Error
c3 = axes[1, 0].contourf(X, Y, error, levels=50, cmap='hot_r')
plt.colorbar(c3, ax=axes[1, 0])
axes[1, 0].set_title(f'Absolute Error (max = {error.max():.2e})', fontsize=14)
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('y')
axes[1, 0].set_aspect('equal')

# Plot 4: Loss History
axes[1, 1].semilogy(history['total'], label='Total', alpha=0.8)
axes[1, 1].semilogy(history['pde'], label='PDE', alpha=0.8)
axes[1, 1].semilogy(history['bc'], label='BC', alpha=0.8)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Training Loss History', fontsize=14)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print accuracy
print(f"Max absolute error:  {error.max():.6e}")
print(f"Mean absolute error: {error.mean():.6e}")
print(f"Relative L2 error:   {np.linalg.norm(u_pred - u_exact) / np.linalg.norm(u_exact):.6e}")


In [ ]:
# ============================================================
# BONUS: 3D Surface Plot Comparison
# ============================================================

fig = plt.figure(figsize=(16, 6))

# PINN Prediction - 3D
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, u_pred, cmap='viridis', alpha=0.8)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_zlabel('u(x,y)')
ax1.set_title('PINN Prediction', fontsize=14)

# Exact Solution - 3D
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(X, Y, u_exact, cmap='viridis', alpha=0.8)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_zlabel('u(x,y)')
ax2.set_title('Exact Solution', fontsize=14)

plt.tight_layout()
plt.show()


## Key Takeaways

1. **2D domains need more points**: We used 5,000 interior + 400 boundary points.

2. **Four boundaries**: Each edge of the square requires separate boundary points.

3. **Residual connections** help deeper networks converge for 2D problems.

4. **The Laplacian** $\nabla^2 u = \partial^2 u/\partial x^2 + \partial^2 u/\partial y^2$ 
   requires computing second derivatives in BOTH x and y directions.

5. **Visualization**: 2D problems produce beautiful contour and surface plots.

### Exercises
- Try different source terms $f(x,y)$
- Try non-zero boundary conditions
- Try an L-shaped domain (requires more careful boundary point placement)
- Move on to **Project 4: Inverse Problem — Parameter Identification**
